# Notebook 01: Data acquisition and inventory

**Project:** Offline On-Device System for Cassava and Maize Disease Detection Using CNNs
**Framework component:** F2, data preparation, steps 1 to 3 of the data-preparation procedure

This notebook:

1. downloads the four selected releases (D1 Kaggle cassava, D3 CCMT Ghana raw images, D4 Tanzania maize v6.0, D6 PlantVillage maize, unaugmented);
2. checks each download against the checksum the publisher gives, where one exists, and records a SHA-256 for every archive;
3. opens **every** image, records its hashes, size, format and whether it decodes, and applies the label map in the label map;
4. saves a working copy of each selected image (EXIF orientation applied, RGB, longest side at most 512 px, JPEG quality 95). The originals are never changed;
5. writes the inventory, a count table (downloaded, readable, selected and excluded per source and label) and a release register to Google Drive, and gives you a small results file to send back.

**Runtime:** CPU is enough (Runtime > Change runtime type > CPU). Expect about 1 to 1.5 hours, mostly downloading.
**Re-running is safe:** finished steps are skipped, so if Colab disconnects, run all cells again.

### Before you start (once)
1. Sign in to Kaggle, open https://www.kaggle.com/competitions/cassava-leaf-disease-classification/rules and click **I Understand and Accept**.
2. On Kaggle go to **Settings > API > Generate New Token** and copy the token it shows.
3. In Colab, click the **key icon (Secrets)** in the left sidebar and add a secret named `KAGGLE_API_TOKEN` whose value is the token Kaggle shows you, with **Notebook access** switched on. (Older `kaggle.json` credentials also work as two secrets, `KAGGLE_USERNAME` and `KAGGLE_KEY`.) Do not paste the token into any cell.

In [ ]:
# 1. Setup: Google Drive, folders and environment record
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, time, platform, subprocess, datetime, shutil, glob, tarfile, csv
DRIVE = '/content/drive/MyDrive/ocdf'
RAW = '/content/raw'            # downloads (temporary, this session only)
WORK = '/content/working'       # working copies before they are packed to Drive
for d in [DRIVE, f'{DRIVE}/manifests', f'{DRIVE}/working', f'{DRIVE}/records', RAW, WORK]:
    os.makedirs(d, exist_ok=True)

def done(step): return os.path.exists(f'{DRIVE}/records/{step}.done')
def mark(step, info=None):
    with open(f'{DRIVE}/records/{step}.done', 'w') as f: json.dump(info or {}, f, indent=1)

import PIL, numpy
env = {
    'notebook': '01_data_acquisition_and_inventory', 'run_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'python': sys.version.split()[0], 'platform': platform.platform(), 'cpu_count': os.cpu_count(),
    'pillow': PIL.__version__, 'numpy': numpy.__version__,
    'disk_free_gb': round(shutil.disk_usage('/content').free / 1e9, 1),
}
json.dump(env, open(f'{DRIVE}/records/environment_nb01.json', 'w'), indent=1)
print(json.dumps(env, indent=1))

In [ ]:
# 2. Helper functions: hashing, perceptual hash, image inspection, inventory files
# ---- OCDF F2 inventory helpers (Notebook 01) ----
import hashlib, io, os, csv, json
import numpy as np
from PIL import Image, ImageOps

WORKING_MAX_SIDE = 512        # working copies: longest side <= 512 px, aspect kept
WORKING_JPEG_QUALITY = 95
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

_DCT32 = None
def _dct_matrix(n=32):
    # Unnormalised DCT-II, identical to scipy.fftpack.dct (as used by imagehash.phash)
    k = np.arange(n)[:, None]; i = np.arange(n)[None, :]
    return 2.0 * np.cos(np.pi * (2 * i + 1) * k / (2 * n))

def phash_hex(img):
    """64-bit DCT perceptual hash (same method as imagehash.phash, hash_size=8)."""
    global _DCT32
    if _DCT32 is None:
        _DCT32 = _dct_matrix(32)
    g = np.asarray(img.convert("L").resize((32, 32), Image.LANCZOS), dtype=np.float64)
    d = _DCT32 @ g @ _DCT32.T
    low = d[:8, :8]
    bits = (low > np.median(low)).flatten()
    return "%016x" % int("".join("1" if b else "0" for b in bits), 2)

def hamming(h1, h2):
    return bin(int(h1, 16) ^ int(h2, 16)).count("1")

def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

def md5_file(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

def inspect_image(raw_bytes, working_out=None):
    """Decode one original image; return inventory fields. Optionally write a working copy.

    The original bytes are never modified. The working copy is EXIF-orientation corrected,
    RGB, longest side <= WORKING_MAX_SIDE, saved as JPEG.
    """
    rec = {"sha256": sha256_bytes(raw_bytes), "bytes": len(raw_bytes), "decode_ok": False,
           "format": "", "width": "", "height": "", "mode": "", "phash": "", "decode_error": "",
           "working_path": "", "working_width": "", "working_height": ""}
    try:
        with Image.open(io.BytesIO(raw_bytes)) as im:
            rec["format"] = im.format or ""
            rec["width"], rec["height"] = im.size
            rec["mode"] = im.mode
            im.load()                                   # full decode = integrity check
            im = ImageOps.exif_transpose(im)
            rgb = im.convert("RGB")
        rec["phash"] = phash_hex(rgb)
        rec["decode_ok"] = True
        if working_out:
            w = rgb.copy()
            w.thumbnail((WORKING_MAX_SIDE, WORKING_MAX_SIDE), Image.LANCZOS)
            os.makedirs(os.path.dirname(working_out), exist_ok=True)
            w.save(working_out, "JPEG", quality=WORKING_JPEG_QUALITY)
            rec["working_path"] = working_out
            rec["working_width"], rec["working_height"] = w.size
    except Exception as e:                              # corrupt / unreadable file
        rec["decode_error"] = f"{type(e).__name__}: {e}"[:200]
    return rec

INVENTORY_FIELDS = ["source_id", "release", "original_relpath", "original_label", "research_label",
                    "crop", "selected", "exclusion_reason", "sha256", "bytes", "decode_ok", "decode_error",
                    "format", "width", "height", "mode", "phash", "working_path", "working_width",
                    "working_height", "group_hint", "published_checksum", "checksum_match"]

def write_inventory(rows, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=INVENTORY_FIELDS, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow(r)

def count_table(rows):
    """Per source x original label: downloaded, readable, selected (research label), excluded."""
    from collections import defaultdict
    t = defaultdict(lambda: {"downloaded": 0, "readable": 0, "unreadable": 0, "selected": 0, "excluded": 0})
    for r in rows:
        k = (r["source_id"], r["original_label"], r["research_label"] or "(excluded)")
        t[k]["downloaded"] += 1
        ok = str(r["decode_ok"]) == "True"
        t[k]["readable" if ok else "unreadable"] += 1
        sel = str(r["selected"]) == "True"          # CSV round-trip turns booleans into text
        t[k]["selected" if (sel and ok) else "excluded"] += 1
    return [dict(source_id=k[0], original_label=k[1], research_label=k[2], **v) for k, v in sorted(t.items())]

In [ ]:
# 3. Release register and label map (label map)
RELEASES = {
  'D1': {'title': 'Cassava Leaf Disease Classification', 'creator': 'Makerere University AI Lab (Kaggle competition)',
         'url': 'https://www.kaggle.com/competitions/cassava-leaf-disease-classification', 'licence': 'Kaggle competition rules',
         'role': 'Cassava development'},
  'D3': {'title': 'Dataset for Crop Pest and Disease Detection (CCMT), Raw Data', 'creator': 'Mensah et al.',
         'doi': '10.17632/bwh3zbpkpv.1', 'version': '1', 'url': 'https://data.mendeley.com/datasets/bwh3zbpkpv/1',
         'licence': 'CC BY 4.0', 'role': 'External test only'},
  'D4': {'title': 'Machine learning imagery dataset for maize crop: A case of Tanzania', 'creator': 'Mduma & Laizer',
         'doi': '10.7910/DVN/GDON8Q', 'version': '6.0', 'url': 'https://doi.org/10.7910/DVN/GDON8Q',
         'licence': 'CC0 1.0', 'role': 'Maize development'},
  'D6': {'title': 'PlantVillage, Plant_leave_diseases_dataset_without_augmentation (maize classes)',
         'creator': 'Hughes & Salathe; archive as used by TensorFlow Datasets plant_village 1.0.2',
         'url': 'https://data.mendeley.com/public-files/datasets/tywbtsjrjv/files/d5652a28-c1d8-4b76-97f3-72fb80f94efc/file_downloaded',
         'licence': 'Republication shows CC0; original creator terms recorded separately', 'role': 'Maize development'},
}
# research label for each (source, original label); '' means excluded
LABEL_MAP = {
  ('D1', 'CBB'): ('C01', 'cassava'), ('D1', 'CMD'): ('C02', 'cassava'), ('D1', 'Healthy'): ('C03', 'cassava'),
  ('D1', 'CBSD'): ('', 'cassava'), ('D1', 'CGM'): ('', 'cassava'),
  ('D3', 'Cassava/bacterial blight'): ('C01', 'cassava'), ('D3', 'Cassava/mosaic'): ('C02', 'cassava'),
  ('D3', 'Cassava/healthy'): ('C03', 'cassava'), ('D3', 'Maize/streak virus'): ('M01', 'maize'),
  ('D3', 'Maize/healthy'): ('M05', 'maize'),
  ('D4', 'MSV_1'): ('M01', 'maize'), ('D4', 'MSV_2'): ('M01', 'maize'), ('D4', 'HEATHLY'): ('M05', 'maize'),
  ('D6', 'Corn___Northern_Leaf_Blight'): ('M02', 'maize'),
  ('D6', 'Corn___Cercospora_leaf_spot Gray_leaf_spot'): ('M03', 'maize'),
  ('D6', 'Corn___Common_rust'): ('M04', 'maize'), ('D6', 'Corn___healthy'): ('M05', 'maize'),
}
EXCLUSION_REASON = {('D1', 'CBSD'): 'outside primary task (label map)',
                    ('D1', 'CGM'): 'equivalence with CCMT green mite cannot be confirmed (label map)'}
ARCHIVES = {}   # filled as archives are downloaded: name -> {sha256, bytes, published checksum, match}

In [ ]:
# 4. Parallel inventory of a list of (relpath, original_label, file_path, extra) items
from concurrent.futures import ProcessPoolExecutor

def _work(args):
    src, release, relpath, olabel, fpath, extra = args
    rlabel, crop = LABEL_MAP.get((src, olabel), ('', ''))
    with open(fpath, 'rb') as f: raw = f.read()
    sha = sha256_bytes(raw)
    wout = f'{WORK}/{src}/{rlabel}/{sha[:20]}.jpg' if rlabel else None
    rec = inspect_image(raw, wout)
    rec.update(source_id=src, release=release, original_relpath=relpath, original_label=olabel,
               research_label=rlabel, crop=crop, selected=bool(rlabel),
               exclusion_reason='' if rlabel else EXCLUSION_REASON.get((src, olabel), 'label not selected'),
               group_hint=extra.get('group_hint', ''), published_checksum=extra.get('published', ''),
               checksum_match=('' if not extra.get('published') else str(extra['published'] == sha)))
    if rec['working_path']:
        rec['working_path'] = os.path.relpath(rec['working_path'], WORK)
    return rec

def inventory(src, release, items):
    t0 = time.time()
    import multiprocessing as mp
    with ProcessPoolExecutor(max_workers=os.cpu_count(), mp_context=mp.get_context('fork')) as ex:
        rows = list(ex.map(_work, [(src, release, *it) for it in items], chunksize=64))
    ok = sum(1 for r in rows if r['decode_ok'])
    print(f'{src}: {len(rows)} images inventoried ({ok} readable) in {time.time()-t0:.0f}s')
    return rows

def pack_working(src):
    # one uncompressed tar per source is far faster on Drive than thousands of small files
    tar_path = f'{DRIVE}/working/working_{src}.tar'
    with tarfile.open(tar_path, 'w') as tar:
        tar.add(f'{WORK}/{src}', arcname=src)
    info = {'tar': os.path.basename(tar_path), 'sha256': sha256_file(tar_path), 'bytes': os.path.getsize(tar_path)}
    print('packed', info)
    return info

def finish_source(src, rows, extra_info):
    write_inventory(rows, f'{DRIVE}/manifests/inventory_{src}.csv')
    info = {'archives': extra_info, 'working': pack_working(src), 'n_rows': len(rows)}
    mark(f'inventory_{src}', info)

In [ ]:
# 5. D1: Kaggle cassava (about 6 GB download)
import pandas as pd
if not done('inventory_D1'):
    from google.colab import userdata
    def secret(name):
        try: return userdata.get(name)
        except Exception: return None
    token = secret('KAGGLE_API_TOKEN')            # new-style token ("Generate New Token")
    if token:
        os.environ['KAGGLE_API_TOKEN'] = token.strip()
        for k in ('KAGGLE_USERNAME', 'KAGGLE_KEY'): os.environ.pop(k, None)
        print('Using KAGGLE_API_TOKEN')
    elif secret('KAGGLE_USERNAME') and secret('KAGGLE_KEY'):   # legacy kaggle.json credentials
        os.environ['KAGGLE_USERNAME'] = secret('KAGGLE_USERNAME').strip(); os.environ['KAGGLE_KEY'] = secret('KAGGLE_KEY').strip()
        print('Using legacy KAGGLE_USERNAME / KAGGLE_KEY')
    else:
        raise RuntimeError('No Kaggle credentials found. Add a Colab secret named KAGGLE_API_TOKEN (key icon, left sidebar) '
                           'and switch on Notebook access.')
    d1 = f'{RAW}/D1'; os.makedirs(d1, exist_ok=True)
    zip_path = f'{d1}/cassava-leaf-disease-classification.zip'
    if not os.path.exists(zip_path):
        subprocess.run(['pip', 'install', '-q', '-U', 'kaggle'], check=True)
        r = subprocess.run(['kaggle', 'competitions', 'download', '-c', 'cassava-leaf-disease-classification', '-p', d1],
                           capture_output=True, text=True)
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        if r.returncode != 0:
            msg = (r.stdout + r.stderr).lower()
            hint = ('Kaggle rejected the credentials: check the KAGGLE_API_TOKEN secret value.' if ('401' in msg or 'unauthor' in msg) else
                    'Kaggle says you have not accepted the competition rules with this account, or the account needs phone verification.' if ('403' in msg or 'forbidden' in msg) else
                    'See the Kaggle message printed above.')
            raise RuntimeError('Kaggle download failed. ' + hint)
    arch = {'cassava-leaf-disease-classification.zip': {'sha256': sha256_file(zip_path), 'bytes': os.path.getsize(zip_path),
            'published_checksum': 'none published', 'retrieved_utc': datetime.datetime.now(datetime.timezone.utc).isoformat()}}
    subprocess.run(['unzip', '-q', '-o', zip_path, 'train_images/*', 'train.csv', 'label_num_to_disease_map.json', '-d', d1], check=True)
    num2name = json.load(open(f'{d1}/label_num_to_disease_map.json'))
    print('Released label dictionary:', num2name)
    def short(name):
        for k in ['CBB', 'CBSD', 'CGM', 'CMD']:
            if f'({k})' in name: return k
        if name.strip().lower() == 'healthy': return 'Healthy'
        raise ValueError(f'Unexpected D1 label: {name}')
    df = pd.read_csv(f'{d1}/train.csv')
    items = [(f'train_images/{r.image_id}', short(num2name[str(r.label)]), f'{d1}/train_images/{r.image_id}', {})
             for r in df.itertuples()]
    print('Rows in train.csv:', len(df), '| image files on disk:', len(glob.glob(f'{d1}/train_images/*')))
    rows = inventory('D1', 'Kaggle competition data', items)
    finish_source('D1', rows, arch)
else:
    print('D1 already done:', json.load(open(f'{DRIVE}/records/inventory_D1.done'))['n_rows'], 'rows')

In [ ]:
# 6. D4: Tanzania maize v6.0 from Harvard Dataverse (3 zips, about 3.2 GB; MLN is not downloaded)
import requests, hashlib, zipfile
D4_FILES = {  # Dataverse file id and the MD5 the repository publishes
  'HEATHLY.zip': (6966997, '2d38c9aac0bfb3dfd5ab8ac832a7110a'),
  'MSV_1.zip':   (6966998, '7aaf5830b95c108563c1e1e275f1886a'),
  'MSV_2.zip':   (6962930, '645c54bbe8bdaeb4288336a005483231'),
}
def download(url, out, tries=5):
    for t in range(tries):
        try:
            with requests.get(url, stream=True, timeout=120) as r:
                r.raise_for_status()
                with open(out + '.part', 'wb') as f:
                    for blk in r.iter_content(1 << 20): f.write(blk)
            os.replace(out + '.part', out); return
        except Exception as e:
            print(f'  retry {t+1} for {os.path.basename(out)}: {e}'); time.sleep(5 * (t + 1))
    raise RuntimeError(f'Download failed: {url}')

if not done('inventory_D4'):
    d4 = f'{RAW}/D4'; os.makedirs(d4, exist_ok=True); arch = {}; items = []
    for name, (fid, md5_pub) in D4_FILES.items():
        zp = f'{d4}/{name}'
        if not os.path.exists(zp):
            print('downloading', name); download(f'https://dataverse.harvard.edu/api/access/datafile/{fid}', zp)
        md5 = md5_file(zp)
        arch[name] = {'dataverse_file_id': fid, 'sha256': sha256_file(zp), 'md5': md5, 'published_md5': md5_pub,
                      'md5_match': md5 == md5_pub, 'bytes': os.path.getsize(zp),
                      'retrieved_utc': datetime.datetime.now(datetime.timezone.utc).isoformat()}
        print(name, 'MD5 matches published value:', md5 == md5_pub)
        assert md5 == md5_pub, f'{name}: MD5 does not match the published value; download again'
        stem = name[:-4]; out = f'{d4}/{stem}'
        if not os.path.exists(out):
            with zipfile.ZipFile(zp) as z: z.extractall(out)
        for p in sorted(glob.glob(f'{out}/**/*', recursive=True)):
            if os.path.isfile(p) and os.path.splitext(p)[1].lower() in IMG_EXT and '__MACOSX' not in p:
                rel = os.path.relpath(p, d4)
                items.append((rel, stem, p, {'group_hint': os.path.dirname(rel)}))
        others = [p for p in glob.glob(f'{out}/**/*', recursive=True) if os.path.isfile(p)
                  and os.path.splitext(p)[1].lower() not in IMG_EXT]
        print(f'  {stem}: {sum(1 for i in items if i[1]==stem)} image files; {len(others)} non-image files ignored')
    rows = inventory('D4', 'Dataverse version 6.0', items)
    finish_source('D4', rows, arch)
else:
    print('D4 already done:', json.load(open(f'{DRIVE}/records/inventory_D4.done'))['n_rows'], 'rows')

In [ ]:
# 7. D3: CCMT raw images from Mendeley Data (5 folders, about 6,200 files; each file has a published SHA-256)
from concurrent.futures import ThreadPoolExecutor
D3_FOLDERS = {
  'Cassava/bacterial blight': '5a089bd5-4c26-415c-862d-530233120bd9',
  'Cassava/healthy':          '17e79247-49e5-4c6b-b81c-11c97599561d',
  'Cassava/mosaic':           'fbf17af1-ab7c-495b-9c9a-705025a7eadc',
  'Maize/healthy':            'b8b2a5ff-7f79-46c4-8815-ba4cec23cfa1',
  'Maize/streak virus':       'd871ac9d-2d79-425f-9d4a-c5ad0de7800a',
}
API = 'https://data.mendeley.com/public-api/datasets/bwh3zbpkpv'
if not done('inventory_D3'):
    d3 = f'{RAW}/D3'; listing = []
    for label, fid in D3_FOLDERS.items():
        start = 0
        while True:
            r = requests.get(f'{API}/files', params={'folder_id': fid, 'version': 1, '$start': start, '$limit': 1000}, timeout=60)
            r.raise_for_status(); page = r.json(); start += len(page)
            for f in page:
                listing.append((label, f['filename'], f['content_details']['download_url'], f['content_details']['sha256_hash']))
            if len(page) < 1000: break
        print(f'{label}: {sum(1 for x in listing if x[0]==label)} files listed')
    json.dump(listing, open(f'{DRIVE}/records/D3_file_listing.json', 'w'))
    def fetch(x):
        label, fname, url, sha_pub = x
        out = f'{d3}/{label}/{fname}'
        if not (os.path.exists(out) and sha256_file(out) == sha_pub):
            os.makedirs(os.path.dirname(out), exist_ok=True); download(url, out)
        return out
    t0 = time.time()
    with ThreadPoolExecutor(16) as ex: paths = list(ex.map(fetch, listing))
    print(f'downloaded {len(paths)} files in {time.time()-t0:.0f}s')
    items = [(f'Raw Data/CCMT Dataset/{lab}/{fn}', lab, p, {'published': sha})
             for (lab, fn, url, sha), p in zip(listing, paths)]
    rows = inventory('D3', 'Mendeley Data version 1', items)
    bad = [r for r in rows if r['checksum_match'] != 'True']
    print('files whose SHA-256 does not match Mendeley:', len(bad))
    finish_source('D3', rows, {'per_file_sha256_checked': True, 'mismatches': len(bad),
                               'retrieved_utc': datetime.datetime.now(datetime.timezone.utc).isoformat()})
else:
    print('D3 already done:', json.load(open(f'{DRIVE}/records/inventory_D3.done'))['n_rows'], 'rows')

In [ ]:
# 8. D6: PlantVillage unaugmented archive (same URL, folder and labels as the TFDS plant_village 1.0.2 builder)
D6_URL = RELEASES['D6']['url']
D6_LABELS = ['Corn___Cercospora_leaf_spot Gray_leaf_spot', 'Corn___Common_rust', 'Corn___healthy', 'Corn___Northern_Leaf_Blight']
if not done('inventory_D6'):
    d6 = f'{RAW}/D6'; os.makedirs(d6, exist_ok=True); zp = f'{d6}/plantvillage.zip'
    if not os.path.exists(zp):
        print('downloading PlantVillage archive'); download(D6_URL, zp)
    arch = {'plantvillage.zip': {'sha256': sha256_file(zp), 'bytes': os.path.getsize(zp),
            'published_checksum': 'see TFDS checksums file for plant_village', 'retrieved_utc': datetime.datetime.now(datetime.timezone.utc).isoformat()}}
    with zipfile.ZipFile(zp) as z:
        names = [n for n in z.namelist() if '/Plant_leave_diseases_dataset_without_augmentation/' in '/' + n]
        items = []
        for lab in D6_LABELS:
            # TFDS matches a space or an underscore in folder names; do the same
            import re
            pat = re.compile(r'(^|/)Plant_leave_diseases_dataset_without_augmentation/' + re.escape(lab).replace(r'\ ', '[_ ]') + r'/[^/]+\.[jJ][pP][gG]$')
            sel = [n for n in names if pat.search(n)]
            print(f'{lab}: {len(sel)} images')
            assert sel, f'No images found for {lab}; check the archive structure'
            for n in sel:
                out = f'{d6}/x/{n}'
                if not os.path.exists(out):
                    os.makedirs(os.path.dirname(out), exist_ok=True)
                    with z.open(n) as s, open(out, 'wb') as t: shutil.copyfileobj(s, t)
                items.append((n, lab, out, {}))
    rows = inventory('D6', 'TFDS plant_village 1.0.2 selection', items)
    finish_source('D6', rows, arch)
else:
    print('D6 already done:', json.load(open(f'{DRIVE}/records/inventory_D6.done'))['n_rows'], 'rows')

In [ ]:
# 9. Combine: count table , exact duplicates, release register and results file
import pandas as pd
srcs = ['D1', 'D3', 'D4', 'D6']
inv = pd.concat([pd.read_csv(f'{DRIVE}/manifests/inventory_{s}.csv', dtype=str, keep_default_na=False) for s in srcs])
inv.to_csv(f'{DRIVE}/manifests/inventory_all_v1.csv', index=False)
counts = pd.DataFrame(count_table(inv.to_dict('records')))
counts.to_csv(f'{DRIVE}/manifests/counts_by_source_label_v1.csv', index=False)

ok = inv[inv.decode_ok == 'True']
dup_within = ok[ok.duplicated(['source_id', 'sha256'], keep=False)]
dev = set(ok[ok.source_id.isin(['D1', 'D4', 'D6'])].sha256); ext = set(ok[ok.source_id == 'D3'].sha256)
cross = sorted(dev & ext)
dups = {'exact_duplicate_files_within_sources': dup_within.groupby('source_id').size().to_dict(),
        'exact_duplicate_groups_within_sources': dup_within.groupby('source_id').sha256.nunique().to_dict(),
        'files_identical_between_development_and_D3': len(cross)}

register = {}
for s in srcs:
    info = json.load(open(f'{DRIVE}/records/inventory_{s}.done'))
    register[s] = {**RELEASES[s], **info}
json.dump(register, open(f'{DRIVE}/records/release_register_v1.json', 'w'), indent=1)

summary = {'environment': env, 'counts': counts.to_dict('records'), 'duplicates': dups,
           'checksums': {s: register[s]['archives'] for s in srcs}}
json.dump(summary, open(f'{DRIVE}/records/nb01_summary.json', 'w'), indent=1)
print(counts.to_string(index=False)); print(json.dumps(dups, indent=1))

# Small results file for Claude (no images): download it and leave it in your Downloads folder
res = '/content/nb01_results.zip'
with zipfile.ZipFile(res, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in [f'{DRIVE}/records/nb01_summary.json', f'{DRIVE}/records/release_register_v1.json',
              f'{DRIVE}/records/environment_nb01.json', f'{DRIVE}/manifests/counts_by_source_label_v1.csv',
              f'{DRIVE}/manifests/inventory_all_v1.csv']:
        z.write(p, os.path.basename(p))
from google.colab import files
files.download(res)

### When it finishes
The run records are written to Google Drive under `ocdf/records/` and also downloaded as `nb01_results.zip`. The copies used for the published results are in this repository under `records/nb01/`.